# CIFAR-10 Autoencoder Training Pipeline

Complete GPU-accelerated training pipeline for Google Colab.

## 📋 Quick Setup Guide
1. **Runtime**: Select `Runtime → Change runtime type → GPU (T4)`
2. **Execute**: Run all cells sequentially
3. **Duration**: ~3-5 minutes total

---

## 🚀 Step 1: Clone Repository & Download Dataset

In [ ]:
# Clone repository from GitHub
!git clone https://github.com/NTHung2034/PP-Final_Project.git
%cd PP-Final_Project

## 📊 Step 2: Download CIFAR-10 Dataset

In [ ]:
# Download CIFAR-10 binary dataset
!bash ./scripts/download_cifar10.sh

## 🔨 Step 3: Build CPU Version

In [ ]:
# Create build directory and compile CPU version
!mkdir -p build
%cd build

# Configure and build (CPU only)
!cmake ..
!make -j$(nproc)

## ✅ Step 4: Test CPU Implementation (Optional)

Run unit tests to verify correctness. Skip if you want to proceed directly to training.

In [ ]:
# Optional: Run CPU tests
!./test_dataloader
!./test_layers
!./test_autoencoder

## 🚀 Step 5: Train CPU Version

CPU training for baseline comparison (~15-20 min/epoch on Colab).

In [ ]:
# Train CPU version (baseline)
import time

cpu_start = time.time()
!./train_cpu
cpu_end = time.time()

cpu_time = cpu_end - cpu_start
print(f"\n{'='*60}")
print(f"CPU Training Time: {cpu_time:.2f} seconds ({cpu_time/60:.2f} minutes)")
print(f"{'='*60}")

# Save for comparison
with open('../cpu_training_time.txt', 'w') as f:
    f.write(str(cpu_time))

## 🔨 Step 6: Build GPU Version

In [ ]:
# Clean and rebuild with CUDA enabled
%cd ..
!rm -rf build
!mkdir build
%cd build

!cmake .. -DENABLE_CUDA=ON
!make -j$(nproc)

## ✅ Step 7: Test GPU Implementation (Optional)

Verify GPU kernels are working correctly.

In [ ]:
# Optional: Run GPU tests
!./test_layers_gpu
!./test_autoencoder_gpu

## 🚀 Step 8: Train GPU Version

GPU-accelerated training (expected: 2-5 minutes on T4).

In [ ]:
# Train GPU version
import time

gpu_start = time.time()
!./train_gpu
gpu_end = time.time()

gpu_time = gpu_end - gpu_start
print(f"\n{'='*60}")
print(f"GPU Training Time: {gpu_time:.2f} seconds ({gpu_time/60:.2f} minutes)")
print(f"{'='*60}")

# Save for comparison
with open('../gpu_training_time.txt', 'w') as f:
    f.write(str(gpu_time))

## 📊 Step 9: Performance Comparison

In [ ]:
# Compare CPU vs GPU performance
import os

if os.path.exists('../cpu_training_time.txt') and os.path.exists('../gpu_training_time.txt'):
    with open('../cpu_training_time.txt', 'r') as f:
        cpu_time = float(f.read().strip())
    with open('../gpu_training_time.txt', 'r') as f:
        gpu_time = float(f.read().strip())
    
    speedup = cpu_time / gpu_time
    
    print(f"\n{'='*60}")
    print(f"PERFORMANCE COMPARISON")
    print(f"{'='*60}")
    print(f"CPU Time:     {cpu_time:8.2f} seconds ({cpu_time/60:6.2f} minutes)")
    print(f"GPU Time:     {gpu_time:8.2f} seconds ({gpu_time/60:6.2f} minutes)")
    print(f"{'='*60}")
    print(f"🚀 GPU Speedup: {speedup:.2f}×")
    print(f"{'='*60}\n")
else:
    print("⚠️  Training times not found. Run both CPU and GPU training first.")

## 📈 Step 10: Visualize Results

In [ ]:
# Visualize performance comparison
import matplotlib.pyplot as plt
import os

if os.path.exists('../cpu_training_time.txt') and os.path.exists('../gpu_training_time.txt'):
    with open('../cpu_training_time.txt', 'r') as f:
        cpu_time = float(f.read().strip())
    with open('../gpu_training_time.txt', 'r') as f:
        gpu_time = float(f.read().strip())
    
    speedup = cpu_time / gpu_time
    
    # Create comparison chart
    fig, ax = plt.subplots(figsize=(10, 6))
    platforms = ['CPU', 'GPU']
    times = [cpu_time/60, gpu_time/60]
    colors = ['#ff6b6b', '#4ecdc4']
    
    bars = ax.bar(platforms, times, color=colors, alpha=0.8, edgecolor='black', linewidth=2)
    ax.set_ylabel('Training Time (minutes)', fontsize=12, fontweight='bold')
    ax.set_title('CPU vs GPU Training Performance', fontsize=14, fontweight='bold')
    ax.set_ylim(0, max(times) * 1.2)
    ax.grid(True, alpha=0.3, axis='y')
    
    # Add value labels on bars
    for bar, time in zip(bars, times):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{time:.2f} min',
                ha='center', va='bottom', fontsize=12, fontweight='bold')
    
    # Add speedup annotation
    ax.text(0.5, max(times) * 1.1, f'Speedup: {speedup:.2f}×',
            ha='center', fontsize=16, fontweight='bold',
            bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.7))
    
    plt.tight_layout()
    plt.savefig('../performance_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("✅ Chart saved to 'performance_comparison.png'")
else:
    print("⚠️  Run both CPU and GPU training first.")

## 💾 Step 11: Download Results

In [ ]:
# Package results for download
%cd ..

!tar -czf results.tar.gz \
    models/saved_weights/ \
    *.png \
    *_training_time.txt \
    2>/dev/null

print("\n✅ Results packaged: results.tar.gz")
print("\nDownload via:")
print("  • Files panel (left sidebar) → Right-click → Download")
print("  • Or run the next cell for direct download")

In [ ]:
# Direct download
from google.colab import files
files.download('results.tar.gz')

---

## 📝 Summary

**Pipeline Steps:**
1. Clone repo & download CIFAR-10
2. Build & test CPU version
3. Train CPU (baseline)
4. Build & test GPU version  
5. Train GPU (accelerated)
6. Compare performance
7. Download results

**Expected Results:**
- Training time: CPU ~15-20 min/epoch, GPU ~20-40 sec/epoch
- Speedup: >20× (target), typically 30-50× on T4/V100
- Final loss: ~0.02-0.03 (good reconstruction)

**Next Steps (Phase 4):**
- Extract features using trained encoder
- Train SVM classifier with LIBSVM
- Evaluate classification accuracy

**Troubleshooting:**
- GPU not available: `Runtime → Change runtime type → GPU`
- Out of memory: Reduce batch size in `include/config.h`
- Build errors: Check CUDA compatibility

**Documentation:** See `docs/` folder for detailed guides